In [143]:
!pip install -U langchain
!pip install langchain-community
!pip install langchain-google-genai
!pip install chromadb
!pip install sentence-transformers
!pip install googlemaps
!pip install Groq
!pip install langchain-groq

In [144]:
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [145]:
!pip install requests
import requests

In [165]:
from google.colab import userdata
import os

# Groq
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Google Maps
GOOGLE_MAP_API = userdata.get('GOOGLE_MAP_API')

# Weather API
OPENWEATHER_API_KEY = userdata.get('OPENWEATHER_API_KEY')

# Hotel API
SERP_API_KEY = userdata.get('SERP_API_KEY')


os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("All API keys loaded ✅")

All API keys loaded ✅


In [147]:
import json

with open("bengaluru_places_rag.json", "r") as f:
    data = json.load(f)

print("JSON loaded ✅")


JSON loaded ✅


In [148]:
places = data["places"]

documents = []
for place in places:
    content = f"""
Name: {place['name']}
Address: {place['address']}
Rating: {place['rating']} ({place['rating_count']} reviews)
Hours: {place['hours']}
Budget: {place['budget']['range']} — {place['budget']['notes']}
Description: {place['description']}
Highlights: {', '.join(place['highlights'])}
Best for (who): {', '.join(place['tags']['who'])}
Best for (time): {', '.join(place['tags']['time'])}
Best for (vibe): {', '.join(place['tags']['vibe'])}
""".strip()

    documents.append(Document(
    page_content=content,
    metadata={
        "name": place["name"],
        "budget_level": place["budget"]["level"],
        "who": ", ".join(place["tags"]["who"]),
        "time": ", ".join(place["tags"]["time"]),
        "vibe": ", ".join(place["tags"]["vibe"]),
        "rating": place["rating"],
        "address": place["address"],
        "latitude": place["latitude"],
        "longitude": place["longitude"]
    }
))
print(f"✅ {len(documents)} places loaded as documents")

✅ 37 places loaded as documents


In [149]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
vectorstore = Chroma.from_documents(
    documents,
    embeddings,
    persist_directory="./chroma_db"
)

In [150]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
print("✅ ChromaDB vector store ready")

✅ ChromaDB vector store ready


In [204]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.6,
    max_tokens=10000
)
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are Lila, a warm and helpful travel and hangout companion for people in Bengaluru.
Based on the places information below, suggest the most relevant options.
For each place give the name, why it fits the mood, budget, and highlights.
Keep it friendly and concise.

Places information:
{context}

User's mood and request:
{question}

Your suggestions:
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Lila's chain is ready")

✅ Lila's chain is ready


In [152]:
from langchain.tools import tool

@tool
def bangalore_places_tool(query: str):
    """Searches Bangalore places from the RAG database based on user preferences."""

    docs = retriever.invoke(query)

    return "\n\n".join(
        [doc.page_content for doc in docs]
    )

In [153]:
import googlemaps
gmaps = googlemaps.Client(
    key=GOOGLE_MAP_API
)


In [154]:
@tool
def maps_tool(query: str, latitude: float, longitude: float):
    """Finds nearby places using Google Maps."""

    places = gmaps.places_nearby(
        location=(latitude, longitude),
        radius=5000,
        keyword=query
    )

    results = []

    for place in places["results"][:5]:

        name = place.get("name", "N/A")
        rating = place.get("rating", "N/A")
        address = place.get("vicinity", "N/A")

        results.append(
            f"{name} | Rating: {rating} | Address: {address}"
        )

    return "\n".join(results)


In [155]:
@tool
def weather_tool(latitude: float, longitude: float):
    """Gets current weather information using latitude and longitude."""

    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?lat={latitude}"
        f"&lon={longitude}"
        f"&appid={OPENWEATHER_API_KEY}"
        f"&units=metric"
    )

    response = requests.get(url)

    data = response.json()

    if "weather" not in data:
        return f"Weather data not found: {data}"

    weather = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    humidity = data["main"]["humidity"]

    return (
        f"Weather: {weather}, "
        f"Temperature: {temp}°C, "
        f"Humidity: {humidity}%"
    )

In [184]:
from google.colab import userdata
from langchain.tools import tool
import requests

@tool
def serpapi_hotel_tool(
    location: str,
    check_in_date: str,
    check_out_date: str,
    adults: int = 1
):
    """
    Searches hotels using SerpAPI Google Hotels.

    Args:
        location: City or destination name.
        check_in_date: Check-in date in YYYY-MM-DD format.
        check_out_date: Check-out date in YYYY-MM-DD format.
        adults: Number of adults staying.

    Returns:
        List of hotel results including:
        hotel name, rating, price, hotel class, and description.
    """

    url = "https://serpapi.com/search.json"

    params = {
        "engine": "google_hotels",
        "q": f"hotels in {location}",
        "check_in_date": check_in_date,
        "check_out_date": check_out_date,
        "adults": adults,
        "api_key": SERP_API_KEY
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=15
        )

        if response.status_code != 200:
            return f"API Error: {response.text}"

        data = response.json()

        hotels = data.get("properties", [])

        if not hotels:
            return f"No hotels found in {location}"

        hotel_results = []

        for hotel in hotels[:5]:

            hotel_results.append({
                "name": hotel.get("name"),
                "price_per_night": hotel.get(
                    "rate_per_night", {}
                ).get("lowest"),

                "rating": hotel.get("overall_rating"),

                "hotel_class": hotel.get("hotel_class"),

                "description": hotel.get("description"),

                "check_in_time": hotel.get("check_in_time"),

                "check_out_time": hotel.get("check_out_time")
            })

        return hotel_results

    except Exception as e:
        return f"Error occurred: {str(e)}"

In [186]:
tools = [
    bangalore_places_tool,
    maps_tool,
    weather_tool,
    serpapi_hotel_tool
]

In [197]:
from langchain.agents import create_agent
from langchain_core.prompts import PromptTemplate
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are an intelligent AI travel assistant.

    Help users with:
    - Bangalore tourist recommendations
    - Weather information
    - Maps/location guidance
    - Hotel recommendations

    Use tools whenever needed.
    """
)


In [200]:
def get_agent_response_content(agent_instance, user_query: str):
    """
    Invokes the agent and returns only the content of the final message.
    """
    response = agent_instance.invoke({
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    })

    if response and 'messages' in response and len(response['messages']) > 0:
        final_message = response['messages'][-1]
        if hasattr(final_message, 'content'):
            return final_message.content
    return "No response or content found."

print("✅ Helper function `get_agent_response_content` is ready!")

✅ Helper function `get_agent_response_content` is ready!


Now, you can use this function like so to get clean output:

In [206]:
# Example usage with the previous query
query = "Suggest hangout places for adventure mood with my 4 friends"
clean_output = get_agent_response_content(agent, query)
print(clean_output)

Here are a handful of adventure‑focused hang‑out spots around Bangalore that are perfect for you and your 4 friends. All of them are easily reachable from the city and offer a mix of adrenaline, fun, and a chance to bond.

| # | Place | What you’ll do | Why it’s great for a 5‑person crew | Approx. cost per person | When to go | Quick notes |
|---|-------|----------------|------------------------------------|------------------------|------------|-------------|
| 1 | **Dirt Mania Outdoor Adventures** (Bengaluru, Karnataka 562109) | ATV quad‑biking on 6 km or 12 km off‑road trails | 4‑person groups get a full day of high‑speed fun, plus a safety briefing and a sunset ride. | ₹1,000 – ₹1,800 (depending on bike type) | Day trips, best in the late afternoon/evening | 8 AM‑10 PM (Fri‑Sun). Bring water, sunscreen, and a camera. |
| 2 | **Rocksport Bengaluru** (Kanakapura Main Rd, near Bolare) | Overnight tent‑stay, campfire, rope‑courses, group games, and a “rain dance” (if it rains!) | All‑in